# 31 — Species Selection & Gap Analysis Job Submission

Select nationally threatened species (Swiss Red List, BAFU/Info Flora 2016/2022)
that have GNN-SDM suitability scores, then submit the circuit-theory batch job
for all qualifying species.

**Steps**:
1. Load the Swiss national Red Lists (vascular plants + bryophytes) from BAFU xlsx files
2. Cross-reference against the 2,696 GNN-trained species to find nationally threatened species with suitability scores
3. Filter to species with ≥50 presence patches (required for meaningful circuit-theory analysis)
4. Submit the SageMaker Training Job for batch computation

In [1]:
import config
import numpy as np
import pandas as pd
import pickle
import os
from data_utils import load_species_patches, load_patch_data

# Species-to-patch mapping (lightweight — just a dict of sets)
species_patches, species_counts = load_species_patches(config)
print(f'Species with presence patches: {len(species_patches):,}')

# Get GNN species names without loading the full arrays into memory
scores_npz = np.load('gnn_training_output/suitability_scores.npz', mmap_mode='r')
gnn_species = set(k.replace('_', ' ') for k in scores_npz.files)
scores_npz.close()  # free memory — we only needed the file list
print(f'GNN-trained species: {len(gnn_species):,}')

Coordinate uncertainty filter (<=1000m): 22,723,973 -> 3,143,834 records (13.8% retained)
GBIF records: 3,143,834
Species with >= 20 presence patches: 2,696
Species with presence patches: 2,696
GNN-trained species: 2,696


### Load Swiss National Red Lists

The Swiss Red List of vascular plants (2016) and bryophytes (2022) are published
by Info Flora and commissioned by BAFU (Federal Office for the Environment).
These use IUCN categories applied at the **national** level — a species can be
globally Least Concern but nationally Critically Endangered in Switzerland.

This is far more appropriate for Swiss conservation prioritisation than the
global IUCN Red List (which left 86% of our species as NOT_EVALUATED).

In [2]:
# Load BAFU Red List xlsx files
rl_vasc = pd.read_excel('reference/gefaesspflanzen_tracheophyta.xlsx', header=0)
rl_moss = pd.read_excel('reference/anhang-rote-liste-der-moose.xlsx', header=0)

# Fix headers (actual column names are in row 0)
rl_vasc.columns = rl_vasc.iloc[0].values
rl_vasc = rl_vasc.iloc[1:].reset_index(drop=True)

rl_moss.columns = rl_moss.iloc[0].values
rl_moss = rl_moss.iloc[1:].reset_index(drop=True)

# Extract species name + Red List category
vasc = rl_vasc[['Scientific name', 'CAT']].copy()
vasc.columns = ['scientific_name', 'ch_redlist_cat']
vasc['source'] = 'vascular'

moss = rl_moss[['Scientific name', 'CAT']].copy()
moss.columns = ['scientific_name', 'ch_redlist_cat']
moss['source'] = 'bryophyte'

# Combine
redlist = pd.concat([vasc, moss], ignore_index=True)
redlist = redlist.dropna(subset=['scientific_name', 'ch_redlist_cat'])

print(f'Swiss Red List total entries: {len(redlist):,}')
print(f'  Vascular plants: {(redlist["source"]=="vascular").sum():,}')
print(f'  Bryophytes: {(redlist["source"]=="bryophyte").sum():,}')
print()
print('National Red List category distribution:')
print(redlist['ch_redlist_cat'].value_counts().head(10).to_string())

Swiss Red List total entries: 4,266
  Vascular plants: 2,915
  Bryophytes: 1,351

National Red List category distribution:
ch_redlist_cat
LC         2244
NT          603
VU          542
EN          305
CR          217
DD          137
alle LC      46
RE           38
CR (PE)      24
CR(PE)       19


### Cross-reference with GNN-trained species

Match the Swiss Red List against the 2,696 species that have GNN-SDM
suitability scores. Filter to threatened categories (CR, EN, VU, NT)
and require ≥50 presence patches for meaningful circuit-theory analysis.

In [3]:
MIN_PATCHES = 50  # minimum presence patches for circuit-theory

# Match GNN species against Red List
redlist['in_gnn'] = redlist['scientific_name'].isin(gnn_species)
matched = redlist[redlist['in_gnn']].copy()
print(f'GNN species found on Swiss Red List: {len(matched):,}')

# Filter to threatened categories (strict: CR, EN, VU, NT only)
THREATENED_CATS = ['CR', 'EN', 'VU', 'NT']
threatened = matched[matched['ch_redlist_cat'].isin(THREATENED_CATS)].copy()
print(f'Nationally threatened with GNN scores: {len(threatened):,}')
print()
print('Breakdown by category:')
print(threatened['ch_redlist_cat'].value_counts().to_string())
print()

# Add patch counts
threatened['n_patches'] = threatened['scientific_name'].apply(
    lambda sp: len(species_patches.get(sp, set()))
)

# Filter to species with enough patches
gap_species = threatened[threatened['n_patches'] >= MIN_PATCHES].copy()
gap_species = gap_species.sort_values('ch_redlist_cat', 
    key=lambda x: pd.Categorical(x, categories=['CR', 'EN', 'VU', 'NT'], ordered=True))

print(f'\nSpecies with ≥{MIN_PATCHES} patches (suitable for gap analysis): {len(gap_species)}')
print(f'  CR: {(gap_species["ch_redlist_cat"] == "CR").sum()}')
print(f'  EN: {(gap_species["ch_redlist_cat"] == "EN").sum()}')
print(f'  VU: {(gap_species["ch_redlist_cat"] == "VU").sum()}')
print(f'  NT: {(gap_species["ch_redlist_cat"] == "NT").sum()}')
print()

# Show the species
print(gap_species[['scientific_name', 'ch_redlist_cat', 'source', 'n_patches']].to_string(index=False))

GNN species found on Swiss Red List: 1,817
Nationally threatened with GNN scores: 489

Breakdown by category:
ch_redlist_cat
NT    251
VU    170
EN     58
CR     10


Species with ≥50 patches (suitable for gap analysis): 285
  CR: 2
  EN: 20
  VU: 93
  NT: 170

              scientific_name ch_redlist_cat    source  n_patches
          Oenanthe lachenalii             CR  vascular         92
                 Typha minima             CR  vascular         81
                 Iberis amara             EN  vascular        104
          Gladiolus palustris             EN  vascular         65
            Leonurus cardiaca             EN  vascular        306
             Bromus racemosus             EN  vascular         61
              Galium saxatile             EN  vascular        211
                 Rosa gallica             EN  vascular         57
            Galeopsis segetum             EN  vascular         53
                Gagea villosa             EN  vascular         64
      Aristo

### Save species list for the batch job

Export the filtered threatened species list so it can be referenced by
notebook 32 (gap analysis) and included in the report.

In [4]:
output_dir = 'gap_analysis_output'
os.makedirs(output_dir, exist_ok=True)

# Save the species list for downstream notebooks
csv_path = f'{output_dir}/gap_analysis_species_ch_redlist.csv'
gap_species[['scientific_name', 'ch_redlist_cat', 'source', 'n_patches']].to_csv(
    csv_path, index=False
)
print(f'Saved {len(gap_species)} species to {csv_path}')

# Upload to S3 for the training job
import boto3
s3 = boto3.client('s3')
s3_key = 'processed/gap_analysis/gap_analysis_species_ch_redlist.csv'
s3.upload_file(csv_path, config.S3_BUCKET, s3_key)
print(f'Uploaded to s3://{config.S3_BUCKET}/{s3_key}')

Saved 285 species to gap_analysis_output/gap_analysis_species_ch_redlist.csv
Uploaded to s3://km-cas-datalake/processed/gap_analysis/gap_analysis_species_ch_redlist.csv


### Estimate computation time

Each species takes ~30s for circuit-theory computation (solving sparse
linear systems on a 166k-node graph). Estimate total job time.

In [5]:
SECONDS_PER_SPECIES = 30  # empirical from notebook 30 runs
n_species = len(gap_species)
est_hours = n_species * SECONDS_PER_SPECIES / 3600

print(f'Species to compute: {n_species}')
print(f'Estimated time: {est_hours:.1f} hours')
print(f'  (at ~{SECONDS_PER_SPECIES}s per species on ml.c5.2xlarge)')
print()
if est_hours < 0.5:
    print('→ Fast enough to run locally in this notebook')
elif est_hours < 2:
    print('→ Moderate — consider SageMaker job for reliability (spot interruption handling)')
else:
    print('→ Long-running — SageMaker batch job recommended (with checkpointing)')

Species to compute: 285
Estimated time: 2.4 hours
  (at ~30s per species on ml.c5.2xlarge)

→ Long-running — SageMaker batch job recommended (with checkpointing)


### Submit SageMaker Training Job

Submit the gap analysis as a SageMaker Training Job on a spot instance.
The `gap_analysis_job/run_gap_analysis.py` script handles:
- Checkpointing every 25 species (for spot instance resilience)
- SIGTERM handler for graceful shutdown
- Both raw and normalised current flow outputs
- Per-species distributional statistics for aggregation decisions

In [6]:
import sagemaker
from sagemaker.pytorch import PyTorch

session = sagemaker.Session()
role = sagemaker.get_execution_role()

# Check if job already exists/completed
job_name = 'gnn-sdm-gap-analysis-ch-redlist'

# The training job reads all inputs from S3
# Required files: landscape_graph.pkl, species_patches.pkl, suitability_scores.npz
input_s3 = config.S3_PROCESSED + '/gap_analysis_input'

# Ensure input files are on S3
print('Uploading input data to S3...')
s3 = boto3.client('s3')
input_prefix = 'processed/gap_analysis_input'
for fname in ['landscape_graph.pkl', 'species_patches.pkl']:
    s3.upload_file(fname, config.S3_BUCKET, f'{input_prefix}/{fname}')
    print(f'  {fname}')
s3.upload_file(
    'gnn_training_output/suitability_scores.npz',
    config.S3_BUCKET, f'{input_prefix}/suitability_scores.npz'
)
print('  suitability_scores.npz')
# Upload species filter list so the job only computes our threatened species
species_list_fname = 'gap_analysis_species_ch_redlist.csv'
s3.upload_file(
    f'{output_dir}/{species_list_fname}',
    config.S3_BUCKET, f'{input_prefix}/{species_list_fname}'
)
print(f'  {species_list_fname}')
print('Input data ready on S3')

estimator = PyTorch(
    entry_point='run_gap_analysis.py',
    source_dir='gap_analysis_job',
    role=role,
    instance_count=1,
    instance_type='ml.c5.2xlarge',
    framework_version='2.1',
    py_version='py310',
    use_spot_instances=True,
    max_wait=54000,  # 15h max wait (includes spot queue time)
    max_run=43200,   # 12h max run (plenty for ~285 species at 33s each)
    checkpoint_s3_uri=f's3://{config.S3_BUCKET}/checkpoints/gap-analysis/',
    hyperparameters={
        'min-patches': MIN_PATCHES,
        'n-pairs': 6,
        'checkpoint-every': 25,
        'species-list': 'gap_analysis_species_ch_redlist.csv',
    },
    base_job_name=job_name,
)

print(f'\nSubmitting job: {job_name}')
print(f'  Instance: ml.g4dn.xlarge (spot)')
print(f'  Species: {n_species} (min {MIN_PATCHES} patches)')
print(f'  Estimated time: {est_hours:.1f}h')

estimator.fit({'training': input_s3}, wait=False)
print(f'\nJob submitted! Monitor in SageMaker console.')
print(f'Output will be at: s3://{config.S3_BUCKET}/output/{job_name}/')

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploading input data to S3...
  landscape_graph.pkl
  species_patches.pkl
  suitability_scores.npz
  gap_analysis_s